# AUDIT-X FINAL · G · Fourth-round review analyses (CPU, no training)

Uses only saved outputs; nothing is retrained.

| Section | Review point |
|---|---|
| G1 | Three-seed ensembles of AGEF-Net (original and tuned) vs the deep ensemble and 3 × MobileNetV3-S |
| G2 | TOST (±0.03) for the seed-ensemble comparisons |
| G3 | Stacking on frozen independently trained CNNs (logistic regression; learned global weights), fit on half of the calibration split |
| G4 | Gate collapse and collapsed seeds of AGEF-Net |
| G5 | Recall at matched realized FAR (5 %) and partial AUROC (FPR ≤ 0.05) |
| G6 | Nested shortcut design: configuration selected on one split, evaluated on another |
| G7 | Precision and alerts per 10,000 files at 0.1 %, 1 % and 10 % prevalence |
| G8 | Minimum detectable effects and random-effects pooling for C1–C6 |
| G9 | Near-twin calibration images: are their training twins in capped groups? |
| G10 | Original MaleVis run re-scored without VBA in evaluation |
| G3b | Frozen-expert gate: per-sample gate trained on frozen independently trained CNNs |
| G11 | Family-macro AUROC / UFR; C1–C6 under family-macro AUROC |
| G12 | Distinct unknown families; leave-one-family-out jackknife |
| G13 | What ±0.03 AUROC means in UFR at matched FAR |
| G14 | KNN bank-size sensitivity |
| G15 | Tuned AGEF-Net learning-rate sweep |
| G16 | Near-duplicate rule validity (pixel similarity of matched pairs; image grid for a manual check) |

**Inputs:** committed outputs of **FINAL_A2** (MaleVis, Amendment 1), **FINAL_B**, **FINAL_C**, **FINAL_A** (original MaleVis run, G10 only), and the three datasets **melevis, malimg-dataset-25, dumpware10** (G16 only).
Accelerator **None**. ≈ 3–4 h (estimate). Results: `/kaggle/working/auditx_review4/`.

In [ ]:
%%writefile /kaggle/working/auditx_eval_v2.py
# =============================================================================
# AUDIT-X  —  corrected evaluation pipeline (v2)
# =============================================================================
# One script that produces every number the paper needs, from saved artefacts.
#
# What it fixes relative to the original notebooks
#   1. Known data are split per run into  train / val / calib / test  (70/10/10/10).
#      - val   : epoch selection only (known classes only)
#      - calib : operating thresholds only (target FAR)
#      - test  : FAR and closed-set accuracy are REPORTED here, so FAR is a
#                measurement, not a tautology. Training samples are never scored.
#   2. Repeated over several random unknown-class splits (SPLIT_SEEDS); results are
#      mean ± std across splits, with bootstrap CIs and paired tests across splits.
#   3. Scores are computed in float64 from logits; non-saturating scores
#      (MaxLogit, Energy) are reported next to MSP / entropy, plus the fraction of
#      calibration scores tied at the threshold (saturation diagnostic).
#   4. Threshold-free metrics (AUROC, AUPR-out, FPR@95TPR) are primary;
#      ZDR at a calibrated 5% FAR is the operational metric.
#   5. Real baselines: single CNNs, a jointly-trained average ensemble, an
#      independently-trained deep ensemble, a genuine OpenMax (Weibull on distances
#      to class MAVs), Energy, MaxLogit, KNN on penultimate features.
#   6. Four-tag triage is reported with per-tag fractions on test-known and unknown.
#   7. Optional dynamic early exit that is actually conditional: the exit rule uses
#      MobileNet's own confidence only (the gate needs all three experts, so a
#      gate-entropy exit rule can never save compute). Cost is measured with
#      torch.utils.flop_counter and wall-clock timing per expert.
#   8. Classes named in NEVER_UNKNOWN (benign / "Other") are never held out as
#      zero-day, so flagging benign files cannot count as zero-day recall.
#   9. Everything is resumable: per-run scores are saved to .npz and skipped if present.
#
# Adversarial, federated, concept-drift and SOC-stream experiments are NOT here on
# purpose; see the accompanying notes for why they need redesign first.
#
# Usage (Kaggle cell):   !python auditx_eval_v2.py
# Usage (subset):        !python auditx_eval_v2.py --datasets Malimg --seeds 0 1
# Smoke test:            python auditx_eval_v2.py --smoke
# =============================================================================
import argparse, copy, json, math, os, time, warnings, zlib
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from scipy import stats
from scipy.special import logsumexp, softmax
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------------------------------------------------------
# 0. Configuration (edit paths here, or override from the command line)
# -----------------------------------------------------------------------------
CFG = dict(
    DATASETS=OrderedDict(
        Malimg="/kaggle/input/datasets/mohammadmehedimunna/malimg-dataset-25/Malimg_Dataset_25classes",
        MaleVis="/kaggle/input/datasets/mohammadmehedimunna/melevis/MaleVis",
        Dumpware10="/kaggle/working/dumpware10_merged",   # built from dumpware10_images/224/224/{TRAINING,TEST}
    ),
    # matched case-insensitively against class folder names; these are never held out as zero-day
    NEVER_UNKNOWN=["other", "others", "benign", "goodware", "clean", "legitimate"],
    UNKNOWN_RATIO=0.2,
    SPLIT_SEEDS=[0, 1, 2, 3, 4],
    FRACS=(0.70, 0.10, 0.10, 0.10),          # train / val / calib / test (known classes)
    EPOCHS=12,
    BATCH=32,
    LR=1e-3,
    WD=1e-4,
    IMG=224,
    WORKERS=4,
    PRETRAINED=True,
    AMP=True,
    TARGET_FAR=0.05,
    # Models to train per split. Keys are defined in MODEL_SPECS below.
    MODELS=["agef_sharp", "avg_joint", "single_mobilenet", "single_resnet18", "single_effnet_b0"],
    DEEP_ENSEMBLE=True,                        # built post hoc from the three single_* models
    TRIAGE_AXES=("msp", "entropy"),            # the paper's two axes
    KNN_K=50,
    KNN_MAX_TRAIN=6000,
    OPENMAX_TAIL=20,
    OPENMAX_ALPHA=10,
    RUN_DYNAMIC=True,
    DYNAMIC_EXIT_QUANTILES=[0.0, 0.25, 0.5, 0.75, 0.9],
    N_BOOT=10000,
    OUT_DIR="/kaggle/working/auditx_v2",
    # Duplicate-aware splitting (protocol v3): byte-identical or pixel-identical images form one group; a group is never
    # split across train/val/calib/test; groups containing more than one family (label conflicts) are removed; classes that
    # cannot be split (one group > 90 % of the class, or < 10 images left) are excluded from the experiment.
    # Near-identical images (perceptual hashes) are NOT grouped: low-information byte plots share identical dHashes and
    # transitive matching merges whole families; they are reported as a sensitivity analysis (notebook 12b).
    GROUPING=True,
    GROUP_KEYS=(0, 1),            # 0 = file MD5, 1 = pixel MD5, 2 = 256-bit dHash
    UNSPLITTABLE_SHARE=0.9,
    MIN_CLASS_IMAGES=10,
    # Near-duplicate grouping (protocol v4, applied to the datasets listed): within each family, images whose 256-bit dHash
    # differs by <= NEAR_DUP_BITS are merged, closest pairs first, and a merge is refused if the group would exceed
    # NEAR_DUP_CAP images, which prevents transitive chaining across whole families.
    # cap 250 = smallest cap that brings residual near-identical matches (256-bit dHash within 8 bits) between training and
    # evaluation images below 5 % on every dataset without making a class unsplittable through chaining; chosen from duplicate
    # statistics only, never from detection results.
    NEAR_DUP_DATASETS=("Malimg", "MaleVis", "Dumpware10"),
    # Amendment 1 (MaleVis only): identical 256-bit-hash clusters are merged exactly and never cut by the cap.
    IDENTICAL_WHOLE_DATASETS=("MaleVis",),
    NEAR_DUP_BITS=8,
    NEAR_DUP_CAP=250,
)
PROTOCOL = "grouped-L1-v3"

EXPERT_NAMES = ("mobilenet", "resnet18", "effnet_b0")
MODEL_SPECS = {
    # name                 experts            fusion  temp  lambda_div
    "agef_sharp":        (EXPERT_NAMES,       "gate", 1.0,  0.05),
    "agef_soft":         (EXPERT_NAMES,       "gate", 2.0,  0.05),
    "agef_nodiv":        (EXPERT_NAMES,       "gate", 1.0,  0.0),
    "avg_joint":         (EXPERT_NAMES,       "avg",  1.0,  0.0),
    "single_mobilenet":  (("mobilenet",),     "avg",  1.0,  0.0),
    "single_resnet18":   (("resnet18",),      "avg",  1.0,  0.0),
    "single_effnet_b0":  (("effnet_b0",),     "avg",  1.0,  0.0),
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


# -----------------------------------------------------------------------------
# 1. Data
# -----------------------------------------------------------------------------
def make_transform(img):
    return transforms.Compose([
        transforms.Resize((img, img)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


class IndexedSubset(Dataset):
    """Subset of an ImageFolder with labels remapped to 0..K-1 (unknown -> -1)."""
    def __init__(self, base, indices, remap):
        self.base, self.indices, self.remap = base, np.asarray(indices), remap

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        path, t = self.base.samples[self.indices[i]]
        img = self.base.loader(path)
        if self.base.transform is not None:
            img = self.base.transform(img)
        return img, self.remap.get(t, -1)


def _strat_split(idx, y, test_size, seed):
    try:
        return train_test_split(idx, test_size=test_size, stratify=y, random_state=seed)
    except ValueError:  # a class too small to stratify
        return train_test_split(idx, test_size=test_size, random_state=seed)


def image_keys(path):
    """file MD5, decoded grey-level pixel MD5, and 256-bit dHash (17x16) as hex."""
    import hashlib
    from PIL import Image
    b = open(path, "rb").read()
    with Image.open(path) as im:
        g = im.convert("L")
        pmd5 = hashlib.md5(g.tobytes() + str(g.size).encode()).hexdigest()
        a = np.asarray(g.resize((17, 16), Image.BILINEAR), dtype=np.int16)
    return hashlib.md5(b).hexdigest(), pmd5, np.packbits((a[:, 1:] > a[:, :-1]).flatten()).tobytes().hex()


def compute_groups(base, cache_path=None, key_types=(0, 1), unsplittable_share=0.9, min_class_images=10,
                   near_bits=None, near_cap=50):
    """Union of exact-key equalities (+ optional capped within-family near-duplicate merging) -> duplicate groups.
    Returns dict(groups, drop, exclude_classes, stats, protocol)."""
    keys = None
    if cache_path and os.path.exists(cache_path):
        c = json.load(open(cache_path))
        if c.get("n") == len(base.samples): keys = c["keys"]
    if keys is None:
        keys = []
        for i, (p, _) in enumerate(base.samples):
            keys.append(image_keys(p))
            if (i + 1) % 5000 == 0: print(f"    hashed {i + 1}/{len(base.samples)}", flush=True)
        if cache_path:
            os.makedirs(os.path.dirname(cache_path), exist_ok=True)
            json.dump(dict(n=len(base.samples), keys=keys), open(cache_path, "w"))
    n = len(keys); parent = np.arange(n)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for t in key_types:
        first = {}
        for i, k in enumerate(keys):
            j = first.setdefault(k[t], i)
            if j != i:
                ri, rj = find(i), find(j)
                if ri != rj: parent[ri] = rj
    targets = np.array([t for _, t in base.samples])
    n_near_merges = 0
    if near_bits is not None:
        POP = np.array([bin(i).count("1") for i in range(256)], np.uint8)
        hb = np.stack([np.frombuffer(bytes.fromhex(k[2]), dtype=np.uint8) for k in keys])
        size = np.zeros(n, int)
        for i in range(n): size[find(i)] += 1
        edges = []
        for c in np.unique(targets):
            idx = np.where(targets == c)[0]
            for a0 in range(0, len(idx), 256):
                q = idx[a0:a0 + 256]
                d = POP[hb[q][:, None, :] ^ hb[idx][None, :, :]].sum(-1)
                ii, jj = np.where(d <= near_bits)
                for u, v in zip(ii, jj):
                    if q[u] < idx[v]: edges.append((int(d[u, v]), int(q[u]), int(idx[v])))
        edges.sort()
        for _, i, j in edges:
            ri, rj = find(i), find(j)
            if ri != rj and size[ri] + size[rj] <= near_cap:
                parent[ri] = rj; size[rj] += size[ri]; n_near_merges += 1
    roots = np.array([find(i) for i in range(n)])
    _, groups = np.unique(roots, return_inverse=True)
    fam_per_group = pd.Series(targets).groupby(groups).nunique()
    conflict = fam_per_group[fam_per_group > 1].index.values
    drop = np.isin(groups, conflict)
    sizes = np.bincount(groups)
    exclude = []
    for c in range(len(base.classes)):
        m = (targets == c) & ~drop
        if m.sum() < min_class_images or (m.any() and np.bincount(groups[m]).max() > unsplittable_share * m.sum()):
            exclude.append(c)
    stats = dict(n_images=n, n_groups=int(len(sizes)), n_images_in_multi_groups=int((sizes[groups] > 1).sum()),
                 largest_group=int(sizes.max()), n_conflict_groups=int(len(conflict)), n_dropped=int(drop.sum()),
                 dropped_by_class={base.classes[c]: int(k) for c, k in zip(*np.unique(targets[drop], return_counts=True))},
                 excluded_classes=[base.classes[c] for c in exclude],
                 largest_group_share_by_class={base.classes[c]: round(float(np.bincount(groups[(targets == c) & ~drop]).max()
                                               / max(((targets == c) & ~drop).sum(), 1)), 3) if ((targets == c) & ~drop).any() else None
                                               for c in range(len(base.classes))},
                 near_dup=None if near_bits is None else dict(bits=near_bits, cap=near_cap, merges=n_near_merges))
    prot = PROTOCOL + ("+identical-hash-whole" if 2 in key_types else "") + \
           ("" if near_bits is None else f"+near-L3-{near_bits}bit-cap{near_cap}")
    return dict(groups=groups, drop=drop, exclude_classes=exclude, stats=stats, protocol=prot)


def make_split(base, seed, cfg, G=None):
    """Class-disjoint known/unknown split + 4-way split of known data (70/10/10/10).
    With G (from compute_groups): duplicate groups are kept within one part and label-conflict groups are removed."""
    classes = base.classes
    targets = np.array([t for _, t in base.samples])
    never = {n.lower() for n in cfg["NEVER_UNKNOWN"]}
    eligible = [c for c in range(len(classes)) if classes[c].lower() not in never]
    protected = [classes[c] for c in range(len(classes)) if classes[c].lower() in never]
    if protected:
        print(f"  classes always kept known (benign/other): {protected}")
    n_unk = max(1, int(round(len(classes) * cfg["UNKNOWN_RATIO"])))
    rng = np.random.RandomState(seed)
    unk = sorted(rng.choice(eligible, n_unk, replace=False).tolist())
    known = [c for c in range(len(classes)) if c not in unk]
    excluded = [] if G is None else list(G.get("exclude_classes", []))
    if excluded:
        unk = [c for c in unk if c not in excluded]; known = [c for c in known if c not in excluded]
    keep = np.ones(len(targets), bool) if G is None else ~G["drop"]
    known_idx = np.where(np.isin(targets, known) & keep)[0]
    unk_idx = np.where(np.isin(targets, unk) & keep)[0]
    if G is None:
        f_tr, f_va, f_ca, f_te = cfg["FRACS"]
        tr, rest = _strat_split(known_idx, targets[known_idx], 1 - f_tr, seed)
        va, rest2 = _strat_split(rest, targets[rest], (f_ca + f_te) / (f_va + f_ca + f_te), seed)
        ca, te = _strat_split(rest2, targets[rest2], f_te / (f_ca + f_te), seed)
    else:
        from sklearn.model_selection import StratifiedGroupKFold
        fold = np.full(len(known_idx), -1)
        sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)
        for f, (_, te_) in enumerate(sgkf.split(known_idx, targets[known_idx], G["groups"][known_idx])):
            fold[te_] = f
        tr, va, ca, te = (known_idx[fold <= 6], known_idx[fold == 7], known_idx[fold == 8], known_idx[fold == 9])
        missing = [classes[c] for c in known if not np.isin(c, targets[tr])]
        assert not missing, f"known classes without training images after grouping: {missing}"
        for nm, part in [("val", va), ("calib", ca), ("test", te)]:
            lack = [classes[c] for c in known if not np.isin(c, targets[part])]
            if lack: print(f"  warning: seed {seed}: no {nm} images for {lack} (group sizes)")
    remap = {c: i for i, c in enumerate(known)}
    out = dict(known=known, unknown=unk, remap=remap,
               known_names=[classes[c] for c in known], unknown_names=[classes[c] for c in unk],
               train=tr, val=va, calib=ca, test=te, unk=unk_idx,
               protocol=G.get("protocol", PROTOCOL) if G is not None else "random-v2")
    if G is not None:
        out["n_dropped_conflict"] = int(G["drop"].sum())
        out["excluded_classes"] = [classes[c] for c in excluded]
    return out


def loader(base, idx, remap, cfg, shuffle):
    return DataLoader(IndexedSubset(base, idx, remap), batch_size=cfg["BATCH"], shuffle=shuffle,
                      num_workers=cfg["WORKERS"], pin_memory=torch.cuda.is_available(),
                      persistent_workers=False)


# -----------------------------------------------------------------------------
# 2. Models
# -----------------------------------------------------------------------------
class Expert(nn.Module):
    """Backbone returning penultimate features + its own linear head."""
    def __init__(self, name, num_classes, pretrained):
        super().__init__()
        w = "IMAGENET1K_V1" if pretrained else None
        if name == "mobilenet":
            m = models.mobilenet_v3_small(weights=w); d = m.classifier[3].in_features; m.classifier[3] = nn.Identity()
        elif name == "resnet18":
            m = models.resnet18(weights=w); d = m.fc.in_features; m.fc = nn.Identity()
        elif name == "effnet_b0":
            m = models.efficientnet_b0(weights=w); d = m.classifier[1].in_features; m.classifier[1] = nn.Identity()
        else:
            raise ValueError(name)
        self.name, self.backbone, self.head, self.feat_dim = name, m, nn.Linear(d, num_classes), d

    def forward(self, x):
        f = self.backbone(x)
        return f, self.head(f)


class MultiExpert(nn.Module):
    """AGEF-Net (fusion='gate') or uniform average (fusion='avg'). One expert = single CNN."""
    def __init__(self, expert_names, num_classes, fusion="gate", temp=1.0, pretrained=True):
        super().__init__()
        self.experts = nn.ModuleList([Expert(n, num_classes, pretrained) for n in expert_names])
        self.E, self.fusion, self.temp = len(expert_names), fusion, temp
        if fusion == "gate":
            self.gate = nn.Sequential(nn.Linear(num_classes * self.E, 128), nn.ReLU(), nn.Linear(128, self.E))

    def forward(self, x):
        feats, logits = zip(*[e(x) for e in self.experts])
        stacked = torch.stack(logits, 1)                       # B,E,C
        if self.fusion == "gate":
            w = F.softmax(self.gate(stacked.flatten(1)) / self.temp, dim=1)
        else:
            w = torch.full((x.size(0), self.E), 1.0 / self.E, device=x.device)
        fused = (w.unsqueeze(-1) * stacked).sum(1)
        # per-expert L2 normalisation so no single backbone dominates KNN distances
        return dict(logits=fused, expert_logits=stacked, weights=w,
                    feats=torch.cat([F.normalize(f.float(), dim=1) for f in feats], 1))


# -----------------------------------------------------------------------------
# 3. Training (model selection on known-class val accuracy only)
# -----------------------------------------------------------------------------
def train(model, tr_ld, va_ld, cfg, lambda_div, log_prefix=""):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["LR"], weight_decay=cfg["WD"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["EPOCHS"])
    use_amp = cfg["AMP"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best, best_state, hist = -1.0, None, []
    for ep in range(cfg["EPOCHS"]):
        model.train(); t0 = time.time(); tl, n = 0.0, 0
        for xb, yb in tr_ld:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                out = model(xb)
                loss = F.cross_entropy(out["logits"], yb)
                if lambda_div > 0 and model.E > 1 and model.fusion == "gate":
                    wbar = out["weights"].float().mean(0)
                    loss = loss + lambda_div * (1.0 - (-(wbar * torch.log(wbar + 1e-8)).sum()))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * xb.size(0); n += xb.size(0)
        sched.step()
        acc = evaluate_acc(model, va_ld, use_amp)
        hist.append(dict(epoch=ep, train_loss=tl / n, val_acc=acc, sec=time.time() - t0))
        print(f"{log_prefix} ep {ep:2d} | loss {tl / n:.4f} | val acc {acc:.2f}% | {time.time() - t0:.0f}s", flush=True)
        if acc > best:
            best, best_state = acc, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, best, hist


@torch.no_grad()
def evaluate_acc(model, ld, use_amp=False):
    model.eval(); c = t = 0
    for xb, yb in ld:
        with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
            p = model(xb.to(DEVICE))["logits"].argmax(1).cpu()
        c += (p == yb).sum().item(); t += yb.numel()
    return 100.0 * c / max(t, 1)


@torch.no_grad()
def collect(model, ld):
    """Return float64 numpy outputs for every sample in the loader (fp32 forward)."""
    model.eval(); acc = {k: [] for k in ("logits", "expert_logits", "weights", "feats", "y")}
    for xb, yb in ld:
        out = model(xb.to(DEVICE))
        for k in ("logits", "expert_logits", "weights", "feats"):
            acc[k].append(out[k].float().cpu().numpy().astype(np.float64))
        acc["y"].append(yb.numpy())
    return {k: np.concatenate(v) for k, v in acc.items()}


# -----------------------------------------------------------------------------
# 4. Scores (higher = more likely unknown)
# -----------------------------------------------------------------------------
def entropy(p):
    return -(p * np.log(np.clip(p, 1e-300, 1.0))).sum(-1)


def base_scores(o, is_deep_ensemble=False):
    """o: dict with logits [N,C], expert_logits [N,E,C], weights [N,E]."""
    if is_deep_ensemble:
        pe = softmax(o["expert_logits"], axis=-1)              # N,E,C
        p = pe.mean(1)
        logits = o["expert_logits"].mean(1)
    else:
        logits = o["logits"]; p = softmax(logits, axis=-1)
        pe = softmax(o["expert_logits"], axis=-1)
    s = OrderedDict()
    s["msp"] = -p.max(1)
    s["entropy"] = entropy(p)
    s["maxlogit"] = -logits.max(1)
    s["energy"] = -logsumexp(logits, axis=1)
    if pe.shape[1] > 1:
        s["mutual_info"] = entropy(pe.mean(1)) - entropy(pe).mean(1)
    if (not is_deep_ensemble) and o["weights"].shape[1] > 1 and np.ptp(o["weights"]) > 1e-9:
        s["gate_entropy"] = entropy(o["weights"])
    return s


class KNNScorer:
    """Sun et al. (2022): negative cosine similarity to k-th nearest training feature."""
    def __init__(self, train_feats, k, max_train, seed):
        rng = np.random.RandomState(seed)
        if len(train_feats) > max_train:
            train_feats = train_feats[rng.choice(len(train_feats), max_train, replace=False)]
        self.bank = torch.tensor(train_feats / np.linalg.norm(train_feats, axis=1, keepdims=True), dtype=torch.float32)
        self.k = min(k, len(self.bank))

    def __call__(self, feats):
        q = torch.tensor(feats / np.linalg.norm(feats, axis=1, keepdims=True), dtype=torch.float32)
        out = []
        for i in range(0, len(q), 1024):
            sims = q[i:i + 1024] @ self.bank.T
            out.append(-sims.topk(self.k, dim=1).values[:, -1])
        return torch.cat(out).numpy().astype(np.float64)


class OpenMax:
    """Bendale & Boult (2016) on fused logits: per-class MAV, Weibull fit on the
    largest distances of correctly classified training samples, top-alpha revision."""
    def __init__(self, train_logits, train_y, tail, alpha):
        from scipy.stats import weibull_min
        self.C = train_logits.shape[1]; self.alpha = min(alpha, self.C)
        self.mav = np.zeros((self.C, self.C)); self.params = []
        pred = train_logits.argmax(1)
        for c in range(self.C):
            av = train_logits[(train_y == c) & (pred == c)]
            if len(av) == 0:
                av = train_logits[train_y == c]
            self.mav[c] = av.mean(0)
            d = np.linalg.norm(av - self.mav[c], axis=1)
            t = np.sort(d)[-min(tail, len(d)):]
            if len(t) < 3 or t.std() < 1e-9:
                self.params.append(None)
            else:
                self.params.append(weibull_min.fit(t, floc=0))
        self._cdf = weibull_min.cdf

    def __call__(self, logits):
        N = len(logits); out = np.zeros(N)
        ranks = np.argsort(-logits, axis=1)[:, :self.alpha]
        wts = np.array([(self.alpha - i) / self.alpha for i in range(self.alpha)])
        for n in range(N):
            v = logits[n].copy(); unk = 0.0
            for r, c in enumerate(ranks[n]):
                prm = self.params[c]
                cdf = 0.0 if prm is None else self._cdf(np.linalg.norm(logits[n] - self.mav[c]), *prm)
                om = wts[r] * cdf
                unk += v[c] * om
                v[c] = v[c] * (1 - om)
            z = np.concatenate([[unk], v])
            out[n] = softmax(z)[0]                               # P(unknown)
        return out


# -----------------------------------------------------------------------------
# 5. Metrics
# -----------------------------------------------------------------------------
def fpr_at_95tpr(s_known, s_unk):
    """Known = positive (accepted if score <= thr). FPR = unknowns accepted."""
    thr = np.quantile(s_known, 0.95)
    return float((s_unk <= thr).mean())


def operating_point(s_calib, s_test, s_unk, target_far):
    thr = np.quantile(s_calib, 1 - target_far)
    return dict(
        thr=float(thr),
        zdr=float((s_unk > thr).mean()),
        far_test=float((s_test > thr).mean()),
        far_calib=float((s_calib > thr).mean()),
        tie_frac_calib=float(np.isclose(s_calib, thr, rtol=0, atol=1e-12).mean()),
    )


def threshold_free(s_known_test, s_unk):
    y = np.r_[np.zeros(len(s_known_test)), np.ones(len(s_unk))]
    s = np.r_[s_known_test, s_unk]
    return dict(auroc=float(roc_auc_score(y, s)), aupr_out=float(average_precision_score(y, s)),
                fpr95=fpr_at_95tpr(s_known_test, s_unk))


def four_tag(conf_score, unc_score, tau_c, tau_u):
    """conf_score / unc_score are 'higher = more unknown'. Paper's Eq. 7 mapping:
    0 Auto-block (confident & certain), 1 Quarantine (unconfident & uncertain),
    2 Review (confident & uncertain), 3 Log (unconfident & certain)."""
    lowconf, highunc = conf_score > tau_c, unc_score > tau_u
    return np.where(~lowconf & ~highunc, 0, np.where(lowconf & highunc, 1, np.where(~lowconf & highunc, 2, 3)))


def triage_report(sc_calib, sc_test, sc_unk, axes, target_far):
    a, b = axes
    tau_a = np.quantile(sc_calib[a], 1 - target_far)
    tau_b = np.quantile(sc_calib[b], 1 - target_far)
    tk = four_tag(sc_test[a], sc_test[b], tau_a, tau_b)
    tu = four_tag(sc_unk[a], sc_unk[b], tau_a, tau_b)
    r = dict(tau_a=float(tau_a), tau_b=float(tau_b),
             zdr=float(np.isin(tu, [1, 2]).mean()), far_test=float(np.isin(tk, [1, 2]).mean()),
             wr_test=float(np.isin(tk, [0, 3]).mean()))
    for t, nm in enumerate(["autoblock", "quarantine", "review", "log"]):
        r[f"known_{nm}"] = float((tk == t).mean()); r[f"unk_{nm}"] = float((tu == t).mean())
    return r


# -----------------------------------------------------------------------------
# 6. Dynamic early exit (honest version)
# -----------------------------------------------------------------------------
def measure_expert_costs(model, cfg):
    """FLOPs per image (torch flop counter) and mean latency per 32-image batch, per expert + gate."""
    from torch.utils.flop_counter import FlopCounterMode
    model.eval(); x1 = torch.randn(1, 3, cfg["IMG"], cfg["IMG"], device=DEVICE)
    xb = torch.randn(cfg["BATCH"], 3, cfg["IMG"], cfg["IMG"], device=DEVICE)
    costs = {}
    with torch.no_grad():
        for e in model.experts:
            fc = FlopCounterMode(display=False)
            with fc:
                e(x1)
            for _ in range(3):
                e(xb)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(10):
                e(xb)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            costs[e.name] = dict(gflops=fc.get_total_flops() / 1e9, ms_per_batch=(time.time() - t0) / 10 * 1e3)
    return costs


def dynamic_eval(o_cal, o_test, o_unk, costs, cfg):
    """Exit after MobileNet (expert 0) when its own MSP >= theta; theta chosen as a
    quantile of calib MobileNet-MSP so that a target fraction of known traffic exits.
    The detection score is entropy of whichever output is used; its threshold is
    re-calibrated on calib under the same policy."""
    rows = []
    f_m, f_rest = costs["mobilenet"]["gflops"], sum(v["gflops"] for k, v in costs.items() if k != "mobilenet")
    t_m, t_rest = costs["mobilenet"]["ms_per_batch"], sum(v["ms_per_batch"] for k, v in costs.items() if k != "mobilenet")

    def policy(o, theta):
        pm = softmax(o["expert_logits"][:, 0], axis=-1)
        exit_ = pm.max(1) >= theta
        p = np.where(exit_[:, None], pm, softmax(o["logits"], axis=-1))
        return entropy(p), exit_

    mcal = softmax(o_cal["expert_logits"][:, 0], axis=-1).max(1)
    for q in cfg["DYNAMIC_EXIT_QUANTILES"]:
        theta = np.inf if q == 0 else np.quantile(mcal, 1 - q)
        s_cal, _ = policy(o_cal, theta)
        s_te, ex_te = policy(o_test, theta)
        s_un, ex_un = policy(o_unk, theta)
        op = operating_point(s_cal, s_te, s_un, cfg["TARGET_FAR"])
        exit_rate = np.r_[ex_te, ex_un].mean()
        rows.append(dict(target_exit=q, theta=float(theta), exit_rate_known=float(ex_te.mean()),
                         exit_rate_unknown=float(ex_un.mean()),
                         gflops_per_img=f_m + (1 - exit_rate) * f_rest,
                         gflops_static=f_m + f_rest,
                         est_ms_per_batch=t_m + (1 - exit_rate) * t_rest,
                         ms_static=t_m + t_rest, **op))
    return rows


# -----------------------------------------------------------------------------
# 7. One (dataset, seed) run
# -----------------------------------------------------------------------------
def run_split(ds_name, base, seed, cfg):
    out_dir = os.path.join(cfg["OUT_DIR"], ds_name, f"seed{seed}")
    os.makedirs(out_dir, exist_ok=True)
    sp = make_split(base, seed, cfg, cfg.get("_GROUPS", {}).get(ds_name))
    json.dump({k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in sp.items() if k != "remap"},
              open(os.path.join(out_dir, "split.json"), "w"), indent=1)
    print(f"\n[{ds_name} seed {seed}] known={len(sp['known'])} unknown={sp['unknown_names']} "
          f"| train {len(sp['train'])} val {len(sp['val'])} calib {len(sp['calib'])} test {len(sp['test'])} unk {len(sp['unk'])}")

    lds = {k: loader(base, sp[k], sp["remap"], cfg, shuffle=(k == "train")) for k in ("train", "val")}
    eval_lds = {k: loader(base, sp[k], sp["remap"], cfg, shuffle=False) for k in ("train", "calib", "test", "unk")}
    K = len(sp["known"])
    outputs = {}

    for mname in cfg["MODELS"]:
        npz = os.path.join(out_dir, f"{mname}.npz")
        if os.path.exists(npz):
            d = np.load(npz, allow_pickle=True)
            outputs[mname] = {part: {k: d[f"{part}__{k}"] for k in ("logits", "expert_logits", "weights", "feats", "y")}
                              for part in ("train", "calib", "test", "unk")}
            outputs[mname]["_meta"] = json.loads(str(d["meta"]))
            print(f"  [{mname}] loaded cached outputs"); continue
        experts, fusion, temp, lam = MODEL_SPECS[mname]
        seed_everything(seed * 1000 + zlib.crc32(mname.encode()) % 997)
        model = MultiExpert(experts, K, fusion, temp, cfg["PRETRAINED"]).to(DEVICE)
        model, best_val, hist = train(model, lds["train"], lds["val"], cfg, lam, log_prefix=f"  [{mname}]")
        o = {part: collect(model, ld) for part, ld in eval_lds.items()}
        meta = dict(best_val_acc=best_val, history=hist,
                    test_acc=float(100 * (o["test"]["logits"].argmax(1) == o["test"]["y"]).mean()))
        if cfg["RUN_DYNAMIC"] and fusion == "gate":
            meta["costs"] = measure_expert_costs(model, cfg)
        np.savez_compressed(npz, meta=json.dumps(meta),
                            **{f"{p}__{k}": v for p, d in o.items() for k, v in d.items()})
        o["_meta"] = meta; outputs[mname] = o
        del model
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    singles = [f"single_{n}" for n in EXPERT_NAMES]
    if cfg["DEEP_ENSEMBLE"] and all(s in outputs for s in singles):
        de = {}
        for part in ("train", "calib", "test", "unk"):
            de[part] = dict(
                expert_logits=np.concatenate([outputs[s][part]["expert_logits"] for s in singles], 1),
                feats=np.concatenate([outputs[s][part]["feats"] for s in singles], 1),
                y=outputs[singles[0]][part]["y"])
            de[part]["logits"] = de[part]["expert_logits"].mean(1)
            de[part]["weights"] = np.full((len(de[part]["y"]), 3), 1 / 3)
        p = softmax(de["test"]["expert_logits"], -1).mean(1)
        de["_meta"] = dict(test_acc=float(100 * (p.argmax(1) == de["test"]["y"]).mean()), deep_ensemble=True)
        outputs["deep_ensemble"] = de

    # ---- score + metric every model -------------------------------------------------
    rows, tri_rows, dyn_rows, cls_rows = [], [], [], []
    targets = np.array([t for _, t in base.samples])
    unk_family = np.array([base.classes[t] for t in targets[sp["unk"]]])       # loader order == sp["unk"] order
    for mname, o in outputs.items():
        is_de = mname == "deep_ensemble"
        sc = {part: base_scores(o[part], is_de) for part in ("calib", "test", "unk")}
        knn = KNNScorer(o["train"]["feats"], cfg["KNN_K"], cfg["KNN_MAX_TRAIN"], seed)
        for part in ("calib", "test", "unk"):
            sc[part]["knn"] = knn(o[part]["feats"])
        om = OpenMax(o["train"]["logits"], o["train"]["y"], cfg["OPENMAX_TAIL"], cfg["OPENMAX_ALPHA"])
        for part in ("calib", "test", "unk"):
            sc[part]["openmax"] = om(o[part]["logits"])
        for sname in sc["test"]:
            r = dict(dataset=ds_name, seed=seed, model=mname, score=sname,
                     test_acc=o["_meta"]["test_acc"], n_unknown_classes=len(sp["unknown"]))
            r.update(threshold_free(sc["test"][sname], sc["unk"][sname]))
            r.update(operating_point(sc["calib"][sname], sc["test"][sname], sc["unk"][sname], cfg["TARGET_FAR"]))
            rows.append(r)
        # per-class alarm rate: recall for each held-out family, false-alarm rate for each known class
        test_cls = np.array([sp["known_names"][y] for y in o["test"]["y"]])
        for sname in ("msp", "entropy", "energy", "knn"):
            if sname not in sc["test"]:
                continue
            thr = np.quantile(sc["calib"][sname], 1 - cfg["TARGET_FAR"])
            for fam in np.unique(unk_family):
                m_ = unk_family == fam
                cls_rows.append(dict(dataset=ds_name, seed=seed, model=mname, score=sname, cls=fam, role="unknown",
                                     n=int(m_.sum()), alarm_rate=float((sc["unk"][sname][m_] > thr).mean())))
            for kc in np.unique(test_cls):
                m_ = test_cls == kc
                cls_rows.append(dict(dataset=ds_name, seed=seed, model=mname, score=sname, cls=kc, role="known",
                                     n=int(m_.sum()), alarm_rate=float((sc["test"][sname][m_] > thr).mean())))
        tri = triage_report(sc["calib"], sc["test"], sc["unk"], cfg["TRIAGE_AXES"], cfg["TARGET_FAR"])
        tri_rows.append(dict(dataset=ds_name, seed=seed, model=mname, axes="/".join(cfg["TRIAGE_AXES"]), **tri))
        if cfg["RUN_DYNAMIC"] and "costs" in o["_meta"]:
            for d in dynamic_eval(o["calib"], o["test"], o["unk"], o["_meta"]["costs"], cfg):
                dyn_rows.append(dict(dataset=ds_name, seed=seed, model=mname, **d))
    return rows, tri_rows, dyn_rows, cls_rows


# -----------------------------------------------------------------------------
# 8. Aggregation across splits
# -----------------------------------------------------------------------------
def boot_ci(x, n_boot, seed=0):
    x = np.asarray(x, float)
    if len(x) < 2:
        return (np.nan, np.nan)
    rng = np.random.RandomState(seed)
    m = rng.choice(x, (n_boot, len(x)), replace=True).mean(1)
    return tuple(np.percentile(m, [2.5, 97.5]))


def summarise(raw, cfg):
    metrics = ["auroc", "aupr_out", "fpr95", "zdr", "far_test", "tie_frac_calib", "test_acc"]
    out = []
    for (d, m, s), g in raw.groupby(["dataset", "model", "score"]):
        r = dict(dataset=d, model=m, score=s, n_splits=len(g))
        for k in metrics:
            r[f"{k}_mean"], r[f"{k}_std"] = g[k].mean(), g[k].std(ddof=1) if len(g) > 1 else np.nan
        r["auroc_ci_lo"], r["auroc_ci_hi"] = boot_ci(g["auroc"], cfg["N_BOOT"])
        r["zdr_ci_lo"], r["zdr_ci_hi"] = boot_ci(g["zdr"], cfg["N_BOOT"])
        out.append(r)
    return pd.DataFrame(out)


def paired_tests(raw, cfg, ref=("agef_sharp", "entropy")):
    """Reference (model, score) vs every other (model, score), paired over split seeds."""
    out = []
    for d, gd in raw.groupby("dataset"):
        a = gd[(gd.model == ref[0]) & (gd.score == ref[1])].set_index("seed")
        if a.empty:
            continue
        for (m, s), gb in gd.groupby(["model", "score"]):
            if (m, s) == ref:
                continue
            b = gb.set_index("seed")
            common = a.index.intersection(b.index)
            if len(common) < 2:
                continue
            for k in ("auroc", "zdr"):
                diff = a.loc[common, k].values - b.loc[common, k].values
                t = stats.ttest_rel(a.loc[common, k], b.loc[common, k])
                try:
                    w = stats.wilcoxon(diff).pvalue if np.any(diff != 0) else 1.0
                except ValueError:
                    w = np.nan
                lo, hi = boot_ci(diff, cfg["N_BOOT"])
                out.append(dict(dataset=d, reference=f"{ref[0]}:{ref[1]}", other=f"{m}:{s}", metric=k,
                                n=len(common), mean_diff=diff.mean(), diff_ci_lo=lo, diff_ci_hi=hi,
                                t_p=t.pvalue, wilcoxon_p=w))
    return pd.DataFrame(out)


# -----------------------------------------------------------------------------
# 9. Smoke test data (synthetic, CPU) — verifies the pipeline end to end
# -----------------------------------------------------------------------------
def build_synthetic(root, n_classes=6, n_per=40, img=32, seed=0):
    from PIL import Image
    rng = np.random.RandomState(seed)
    for c in range(n_classes):
        os.makedirs(os.path.join(root, f"fam{c}"), exist_ok=True)
        base = rng.rand(img, img)
        for i in range(n_per):
            a = np.clip(base + 0.25 * rng.randn(img, img), 0, 1)
            Image.fromarray((a * 255).astype(np.uint8), "L").save(os.path.join(root, f"fam{c}", f"{i}.png"))
    os.makedirs(os.path.join(root, "Other"), exist_ok=True)
    for i in range(n_per):
        Image.fromarray((rng.rand(img, img) * 255).astype(np.uint8), "L").save(os.path.join(root, "Other", f"{i}.png"))


# -----------------------------------------------------------------------------
# 10. Main
# -----------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--datasets", nargs="*", help="subset of CFG['DATASETS'] keys")
    ap.add_argument("--path", nargs="*", help="override paths as Name=/path")
    ap.add_argument("--seeds", nargs="*", type=int)
    ap.add_argument("--models", nargs="*")
    ap.add_argument("--epochs", type=int)
    ap.add_argument("--img", type=int)
    ap.add_argument("--batch", type=int)
    ap.add_argument("--workers", type=int)
    ap.add_argument("--out")
    ap.add_argument("--no-pretrained", action="store_true")
    ap.add_argument("--no-dynamic", action="store_true")
    ap.add_argument("--no-grouping", action="store_true", help="old random splits (protocol v2)")
    ap.add_argument("--smoke", action="store_true", help="synthetic CPU run to verify the pipeline")
    ap.add_argument("--resume-from", help="folder with a previous OUT_DIR (e.g. an attached Kaggle output); cached runs are copied and skipped")
    ap.add_argument("--max-hours", type=float, default=None, help="do not start a new split after this many hours")
    args, _ = ap.parse_known_args()

    cfg = copy.deepcopy(CFG)
    if args.smoke:
        root = "/tmp/auditx_smoke_data"
        if not os.path.exists(root):
            build_synthetic(root)
        cfg.update(DATASETS=OrderedDict(Synthetic=root), NEVER_UNKNOWN=["other"], SPLIT_SEEDS=[0, 1], EPOCHS=2, IMG=64, BATCH=16,
                   WORKERS=0, PRETRAINED=False, OUT_DIR="/tmp/auditx_smoke_out", N_BOOT=500,
                   KNN_K=5, OPENMAX_TAIL=5, UNKNOWN_RATIO=0.34)
    if args.path:
        for kv in args.path:
            k, v = kv.split("=", 1); cfg["DATASETS"][k] = v
    if args.datasets:
        cfg["DATASETS"] = OrderedDict((k, cfg["DATASETS"][k]) for k in args.datasets)
    if args.seeds: cfg["SPLIT_SEEDS"] = args.seeds
    if args.models: cfg["MODELS"] = args.models
    if args.epochs: cfg["EPOCHS"] = args.epochs
    if args.img: cfg["IMG"] = args.img
    if args.batch: cfg["BATCH"] = args.batch
    if args.workers is not None: cfg["WORKERS"] = args.workers
    if args.out: cfg["OUT_DIR"] = args.out
    if args.no_pretrained: cfg["PRETRAINED"] = False
    if args.no_dynamic: cfg["RUN_DYNAMIC"] = False
    if args.no_grouping: cfg["GROUPING"] = False

    os.makedirs(cfg["OUT_DIR"], exist_ok=True)
    if args.resume_from and os.path.isdir(args.resume_from):
        import shutil
        n_copied = 0
        for dp, _, fns in os.walk(args.resume_from):
            for fn in fns:
                if fn.endswith(".npz") or fn == "split.json":
                    rel = os.path.relpath(os.path.join(dp, fn), args.resume_from)
                    dst = os.path.join(cfg["OUT_DIR"], rel)
                    if not os.path.exists(dst):
                        os.makedirs(os.path.dirname(dst), exist_ok=True)
                        shutil.copy2(os.path.join(dp, fn), dst); n_copied += 1
        print(f"Resumed {n_copied} cached files from {args.resume_from}")
    t_start = time.time()
    json.dump({k: (list(v.items()) if isinstance(v, OrderedDict) else v) for k, v in cfg.items()},
              open(os.path.join(cfg["OUT_DIR"], "config.json"), "w"), indent=1)
    print(f"Device: {DEVICE} | torch {torch.__version__} | out: {cfg['OUT_DIR']}")

    all_rows, all_tri, all_dyn, all_cls = [], [], [], []
    for ds_name, path in cfg["DATASETS"].items():
        if not path or not os.path.isdir(path):
            print(f"!! skipping {ds_name}: path not found ({path})"); continue
        base = datasets.ImageFolder(path, transform=make_transform(cfg["IMG"]))
        print(f"\n===== {ds_name}: {len(base)} images, {len(base.classes)} classes | protocol {PROTOCOL if cfg['GROUPING'] else 'random-v2'} =====")
        if cfg["GROUPING"]:
            near = ds_name in cfg.get("NEAR_DUP_DATASETS", ())
            G = compute_groups(base, os.path.join(cfg["OUT_DIR"], ds_name, "image_keys.json"), cfg["GROUP_KEYS"],
                               cfg["UNSPLITTABLE_SHARE"], cfg["MIN_CLASS_IMAGES"],
                               cfg["NEAR_DUP_BITS"] if near else None, cfg["NEAR_DUP_CAP"])
            print("  split protocol:", G["protocol"])
            cfg.setdefault("_GROUPS", {})[ds_name] = G
            json.dump(G["stats"], open(os.path.join(cfg["OUT_DIR"], ds_name, "groups_stats.json"), "w"), indent=1)
            print("  duplicate groups:", json.dumps({k: v for k, v in G["stats"].items() if k not in ("dropped_by_class", "largest_group_share_by_class")}),
                  "| dropped (label conflicts):", G["stats"]["dropped_by_class"], "| excluded classes:", G["stats"]["excluded_classes"])
        for seed in cfg["SPLIT_SEEDS"]:
            if args.max_hours and (time.time() - t_start) / 3600 > args.max_hours:
                print(f"!! time budget of {args.max_hours} h reached; not starting seed {seed}. "
                      f"Re-run with --resume-from to continue."); break
            r, t, d, c = run_split(ds_name, base, seed, cfg)
            all_cls += c
            pd.DataFrame(all_cls).to_csv(os.path.join(cfg["OUT_DIR"], "per_class_raw.csv"), index=False)
            print(f"  elapsed {(time.time() - t_start) / 3600:.2f} h", flush=True)
            all_rows += r; all_tri += t; all_dyn += d
            # write after every split so a timed-out session still leaves results
            pd.DataFrame(all_rows).to_csv(os.path.join(cfg["OUT_DIR"], "results_raw.csv"), index=False)
            pd.DataFrame(all_tri).to_csv(os.path.join(cfg["OUT_DIR"], "triage_raw.csv"), index=False)
            if all_dyn:
                pd.DataFrame(all_dyn).to_csv(os.path.join(cfg["OUT_DIR"], "dynamic_raw.csv"), index=False)

    if not all_rows:
        print("No results."); return
    raw = pd.DataFrame(all_rows)
    summ = summarise(raw, cfg); summ.to_csv(os.path.join(cfg["OUT_DIR"], "summary.csv"), index=False)
    paired_tests(raw, cfg).to_csv(os.path.join(cfg["OUT_DIR"], "paired_tests.csv"), index=False)
    tri = pd.DataFrame(all_tri)
    tri.drop(columns=["axes", "seed"]).groupby(["dataset", "model"]).agg(["mean", "std"]).to_csv(os.path.join(cfg["OUT_DIR"], "triage_summary.csv"))
    if all_dyn:
        dyn = pd.DataFrame(all_dyn)
        dyn.groupby(["dataset", "model", "target_exit"]).mean(numeric_only=True).to_csv(
            os.path.join(cfg["OUT_DIR"], "dynamic_summary.csv"))

    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
    show = summ[summ.score.isin(["msp", "entropy", "energy", "knn", "openmax"])][
        ["dataset", "model", "score", "n_splits", "auroc_mean", "auroc_std", "fpr95_mean",
         "zdr_mean", "zdr_std", "far_test_mean", "tie_frac_calib_mean", "test_acc_mean"]]
    print("\n", show.round(4).to_string(index=False))
    print(f"\nAll outputs in {cfg['OUT_DIR']}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/auditx_review4.py
# =============================================================================
# AUDIT-X FINAL - fourth-round review analyses (CPU only, no training)
# Inputs (anywhere under ROOT): auditx_final_malimg/, auditx_final_malevis_v5/ (Amendment 1), auditx_final_dumpware10/,
#   and for G10 only the original MaleVis run auditx_final_malevis/.
#   G1 three-seed ensembles of AGEF-Net (original, tuned) vs deep ensemble and 3 x MobileNetV3-S
#   G2 TOST (+-0.03) for seed-ensemble comparisons
#   G3 stacking on frozen independently trained CNNs (logistic regression; learned global weights), fit on half of calibration
#   G4 gate collapse and collapsed seeds of AGEF-Net
#   G5 UFR at matched realized FAR (5 %) and partial AUROC (FPR <= 0.05)
#   G6 nested shortcut design (select on one split, evaluate on another)
#   G7 precision and alerts per 10,000 files at 0.1 %, 1 %, 10 % prevalence
#   G8 minimum detectable effects and DerSimonian-Laird random-effects pooling for C1-C6
#   G9 near-twin calibration images: group sizes of their training twins (cap) and effect on FAR
#   G10 original MaleVis run re-scored without VBA in evaluation (isolates the VBA exclusion from the grouping change)
#   G3b frozen-expert gate: per-sample gate MLP trained on frozen independently trained CNNs (half of calibration)
#   G11 family-macro AUROC / UFR and C1-C6 under family-macro AUROC;  G12 distinct families and family jackknife
#   G13 what +-0.03 AUROC means in UFR at matched FAR;  G14 KNN bank-size sensitivity;  G15 tuned-AGEF learning-rate sweep
#   G16 near-duplicate rule validity (pixel similarity of matched pairs; needs the datasets attached)
# =============================================================================
import glob, json, os, time, zlib, contextlib, io
import numpy as np, pandas as pd
from scipy.special import softmax, logsumexp, log_softmax
from scipy.stats import rankdata, norm
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import auditx_eval_v2 as ax

ROOT = os.environ.get("AUDITX_ROOT", "/kaggle/input")
OUT = os.environ.get("AUDITX_OUT", "/kaggle/working/auditx_review4"); os.makedirs(OUT, exist_ok=True)
N_BOOT = int(os.environ.get("AUDITX_NBOOT", 1000)); MARGIN = 0.03
DATASETS = ["Malimg", "MaleVis", "Dumpware10"]
MODELS = ["agef_sharp", "agef_tuned", "avg_joint", "single_mobilenet", "single_resnet18", "single_effnet_b0"]
SINGLES = ["single_mobilenet", "single_resnet18", "single_effnet_b0"]
V4 = "grouped-L1-v3+near-L3-8bit-cap250"; V5 = "grouped-L1-v3+identical-hash-whole+near-L3-8bit-cap250"
EXPECTED = {"Malimg": V4, "Dumpware10": V4, "MaleVis": V5}
LOG = []
def say(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); LOG.append(s)

def find(name):
    hits = [os.path.dirname(p) for p in glob.glob(f"{ROOT}/**/{name}/MANIFEST.json", recursive=True)]
    return sorted(hits)[0] if hits else None
RUNS = {}
for ds in DATASETS:
    d = find(f"auditx_final_{ds.lower()}" + ("_v5" if ds == "MaleVis" else ""))
    if d is None: say(f"!! {ds}: run not found"); continue
    man = json.load(open(f"{d}/MANIFEST.json")); assert man["complete"] and man["protocol"] == EXPECTED[ds], f"{ds}: wrong run"
    RUNS[ds] = dict(dir=d, targets=np.load(f"{d}/targets.npy"), classes=json.load(open(f"{d}/classes.json")),
                    splits=sorted(int(p.split("split")[-1]) for p in glob.glob(f"{d}/split*") if os.path.isdir(p)), seeds=man["args"]["train_seeds"])
    say(f"{ds}: {d} | protocol OK")
assert RUNS

def auroc(k, u):
    r = rankdata(np.r_[k, u]); return (r[len(k):].sum() - len(u) * (len(u) + 1) / 2) / (len(u) * len(k))
def cthr(x, a):
    n = len(x); k = int(np.ceil((n + 1) * (1 - a))); return np.inf if k > n else np.sort(x)[k - 1]
def load_npz(p):
    z = np.load(p, allow_pickle=True); parts = {}
    for k in z.files:
        if "__" in k:
            pp, kk = k.split("__", 1)
            if kk in ("logits", "expert_logits", "weights", "feats", "y"): parts.setdefault(pp, {})[kk] = z[k].astype(np.float64) if kk != "y" else z[k]
    return parts, json.loads(str(z["meta"]))
def ens(parts_list):
    out = {}
    for p in ("train", "calib", "test", "unk"):
        el = np.concatenate([o[p]["expert_logits"] if o[p]["expert_logits"].ndim == 3 else o[p]["logits"][:, None] for o in parts_list], 1)
        out[p] = dict(expert_logits=el, logits=el.mean(1), feats=np.concatenate([o[p]["feats"] for o in parts_list], 1), y=parts_list[0][p]["y"])
    return out
def seed_ens(parts_list):
    """average the softmax of whole models (fused logits) over training seeds"""
    out = {}
    for p in ("train", "calib", "test", "unk"):
        el = np.stack([o[p]["logits"] for o in parts_list], 1)
        out[p] = dict(expert_logits=el, logits=el.mean(1), feats=np.concatenate([o[p]["feats"] for o in parts_list], 1), y=parts_list[0][p]["y"])
    return out
def energy(lg): return -logsumexp(lg, axis=1)
def knn(o, k=50, seed=0):
    kn = ax.KNNScorer(o["train"]["feats"], k, 6000, seed); return {p: kn(o[p]["feats"]) for p in ("calib", "test", "unk")}

# ----------------------------------------------------------------------------- load and score
SC, AUX, t0 = {}, {}, time.time()
for ds, R in RUNS.items():
    for s in R["splits"]:
        sp = json.load(open(f"{R['dir']}/split{s}/split.json")); K = len(sp["known"])
        unk_fam = np.array([R["classes"][t] for t in R["targets"][sp["unk"]]])
        per = {}
        for t in R["seeds"]:
            loaded = {m: load_npz(f"{R['dir']}/split{s}/seed{t}/{m}.npz") for m in MODELS}
            outs = {m: v[0] for m, v in loaded.items()}; per[t] = outs
            best = max(SINGLES, key=lambda x: max(h["val_acc"] for h in loaded[x][1]["history"]))   # validation-selected single CNN (C6)
            de = ens([outs[x] for x in SINGLES])
            cfgs = {"deep_ensemble": de, "agef_sharp": outs["agef_sharp"], "agef_tuned": outs["agef_tuned"], "avg_joint": outs["avg_joint"],
                    "single_mobilenet": outs["single_mobilenet"], "single_resnet18": outs["single_resnet18"], "single_effnet_b0": outs["single_effnet_b0"]}
            for m, o in cfgs.items():
                sc = {p: {"energy": energy(o[p]["logits"])} for p in ("calib", "test", "unk")}
                kk = knn(o, 50, s)
                for p in sc: sc[p]["knn50"] = kk[p]
                acc = float(((softmax(o["test"]["expert_logits"], -1).mean(1) if m == "deep_ensemble" else softmax(o["test"]["logits"], -1)).argmax(1) == o["test"]["y"]).mean())
                SC[(ds, s, t, m)] = dict(sc=sc, unk_fam=unk_fam, acc=acc)
            SC[(ds, s, t, "best_single")] = SC[(ds, s, t, best)]
            # G4: gate statistics
            for m in ("agef_sharp", "agef_tuned"):
                w = outs[m]["test"]["weights"]; mw = w.mean(0)
                AUX.setdefault("gate", []).append(dict(dataset=ds, split=s, seed=t, model=m, w_mobilenet=mw[0], w_resnet=mw[1], w_effnet=mw[2],
                                                       max_mean_weight=mw.max(), share_argmax_same=float((w.argmax(1) == np.bincount(w.argmax(1)).argmax()).mean()),
                                                       mean_sample_entropy=float(-(w * np.log(w + 1e-12)).sum(1).mean()), test_acc=SC[(ds, s, t, m)]["acc"],
                                                       auroc_energy=auroc(SC[(ds, s, t, m)]["sc"]["test"]["energy"], SC[(ds, s, t, m)]["sc"]["unk"]["energy"])))
            # G3: stacking on frozen singles, fit on half A of calibration, thresholds on half B
            rng = np.random.RandomState(1000 * s + t); n_c = len(outs[SINGLES[0]]["calib"]["y"]); perm = rng.permutation(n_c); A, Bh = perm[: n_c // 2], perm[n_c // 2:]
            feat = {p: np.concatenate([log_softmax(outs[x][p]["logits"], -1) for x in SINGLES], 1) for p in ("calib", "test", "unk")}
            yA = outs[SINGLES[0]]["calib"]["y"][A]
            try:
                lr = LogisticRegression(max_iter=2000, C=1.0).fit(feat["calib"][A], yA)
                full = np.zeros((1, K)); cls = lr.classes_
                def lr_logits(X):
                    z = np.full((len(X), K), -30.0); z[:, cls] = lr.decision_function(X) if len(cls) > 2 else np.c_[-lr.decision_function(X), lr.decision_function(X)]; return z
                st = {p: lr_logits(feat[p]) for p in ("calib", "test", "unk")}
            except Exception as e:
                st = None; say("stacking failed", ds, s, t, e)
            # learned global weights (softmax-parametrised) minimising calibration-half-A NLL of the weighted logit average
            L = {p: np.stack([outs[x][p]["logits"] for x in SINGLES], 1) for p in ("calib", "test", "unk")}
            def nll(v):
                w = softmax(v); lg = (w[None, :, None] * L["calib"][A]).sum(1); return -log_softmax(lg, -1)[np.arange(len(A)), yA].mean()
            wv = softmax(minimize(nll, np.zeros(3), method="Nelder-Mead").x)
            gw = {p: (wv[None, :, None] * L[p]).sum(1) for p in L}
            # frozen-expert gate: MLP on the concatenated frozen logits -> softmax weights over the 3 experts (per sample)
            import torch
            torch.manual_seed(1000 * s + t)
            Xc = torch.tensor(L["calib"][A], dtype=torch.float32); Yc = torch.tensor(yA, dtype=torch.long)
            gate = torch.nn.Sequential(torch.nn.Linear(3 * K, 64), torch.nn.ReLU(), torch.nn.Linear(64, 3))
            opt = torch.optim.Adam(gate.parameters(), lr=1e-3, weight_decay=1e-4)
            for _ in range(400):
                w = torch.softmax(gate(Xc.flatten(1)), 1); fused = (w.unsqueeze(-1) * Xc).sum(1)
                loss = torch.nn.functional.cross_entropy(fused, Yc); opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                fg = {}
                for p in L:
                    X = torch.tensor(L[p], dtype=torch.float32); w = torch.softmax(gate(X.flatten(1)), 1); fg[p] = (w.unsqueeze(-1) * X).sum(1).numpy().astype(np.float64)
                    if p == "test": AUX.setdefault("fgate", []).append(dict(dataset=ds, split=s, seed=t, mean_w=w.mean(0).numpy().round(3).tolist(),
                                                                          sample_entropy=float(-(w * torch.log(w + 1e-12)).sum(1).mean())))
            for name, lgs in [("stack_lr", st), ("stack_weights", gw), ("frozen_gate", fg), ("deep_ensemble_halfcal", {p: L[p].mean(1) for p in L})]:
                if lgs is None: continue
                sc = {p: {"energy": energy(lgs[p])} for p in ("calib", "test", "unk")}
                sc["calib"] = {"energy": sc["calib"]["energy"][Bh]}
                SC[(ds, s, t, name)] = dict(sc=sc, unk_fam=unk_fam, acc=float((lgs["test"].argmax(1) == outs[SINGLES[0]]["test"]["y"]).mean()),
                                            weights=wv.tolist() if name == "stack_weights" else None)
        # G1: seed ensembles (one evaluation per split)
        for name, mm in [("seedens_agef_sharp", "agef_sharp"), ("seedens_agef_tuned", "agef_tuned"), ("seedens_mobilenet", "single_mobilenet")]:
            o = seed_ens([per[t][mm] for t in R["seeds"]])
            sc = {p: {"energy": energy(o[p]["logits"])} for p in ("calib", "test", "unk")}; kk = knn(o, 50, s)
            for p in sc: sc[p]["knn50"] = kk[p]
            SC[(ds, s, -1, name)] = dict(sc=sc, unk_fam=unk_fam, acc=float((softmax(o["test"]["expert_logits"], -1).mean(1).argmax(1) == o["test"]["y"]).mean()))
        say(f"{ds} split {s} done ({(time.time() - t0) / 60:.1f} min)")

# ----------------------------------------------------------------------------- per-run metrics incl. matched FAR and partial AUROC (G5) and precision (G7)
rows = []
for (ds, s, t, m), d in SC.items():
    if m == "best_single": continue
    for sn, v in d["sc"]["test"].items():
        c, te, u = d["sc"]["calib"][sn], v, d["sc"]["unk"][sn]; th = cthr(c, 0.05); far, ufr = (te > th).mean(), (u > th).mean()
        thm = np.quantile(te, 0.95)                                                     # matched realized FAR = 5 % on test
        y = np.r_[np.zeros(len(te)), np.ones(len(u))]; z = np.r_[te, u]
        r = dict(dataset=ds, split=s, seed=t, model=m, score=sn, auroc=auroc(te, u), pauroc05=roc_auc_score(y, z, max_fpr=0.05),
                 ufr5=ufr, far5=far, ufr_matched5=float((u > thm).mean()), acc=d["acc"])
        for pi in (0.001, 0.01, 0.10):
            den = pi * ufr + (1 - pi) * far; r[f"prec_pi{pi}"] = pi * ufr / den if den > 0 else np.nan; r[f"alerts10k_pi{pi}"] = 1e4 * den
        fams = np.unique(d["unk_fam"])
        r["auroc_family_macro"] = float(np.mean([auroc(te, u[d["unk_fam"] == f]) for f in fams]))
        r["ufr5_family_macro"] = float(np.mean([(u[d["unk_fam"] == f] > th).mean() for f in fams]))
        rows.append(r)
L = pd.DataFrame(rows); L.to_csv(f"{OUT}/G_runs_long.csv", index=False)
SUM = L.groupby(["dataset", "model", "score"]).mean(numeric_only=True).drop(columns=["split", "seed"]).reset_index(); SUM.to_csv(f"{OUT}/G_summary.csv", index=False)

# ----------------------------------------------------------------------------- bootstrap (families, test images, training seeds) and comparisons
def get(ds, m, s, t):
    return SC[(ds, s, -1 if m.startswith("seedens_") else t, m)]
def boot(ds, a, b, sn, n_boot, seed, macro=False, drop_family=None):
    splits, seeds = RUNS[ds]["splits"], RUNS[ds]["seeds"]
    fams = sorted({f for s in splits for f in SC[(ds, s, seeds[0], "agef_sharp")]["unk_fam"]})
    if drop_family is not None: fams = [f for f in fams if f != drop_family]
    def metric(k, u, uf):
        if not macro: return auroc(k, u)
        return np.mean([auroc(k, u[uf == f]) for f in np.unique(uf)])
    def diff(s, t, fidx=None, kb=None):
        A, B = get(ds, a, s, t)["sc"], get(ds, b, s, t)["sc"]
        uf = SC[(ds, s, seeds[0], "agef_sharp")]["unk_fam"]
        if fidx is None:
            fidx = np.where(uf != drop_family)[0] if drop_family is not None else np.arange(len(uf))
        ka, kb_ = (A["test"][sn], B["test"][sn]) if kb is None else (A["test"][sn][kb], B["test"][sn][kb])
        if len(fidx) == 0: return np.nan
        if macro and isinstance(fidx, list):        # bootstrap: one entry per drawn family, with multiplicity
            return np.mean([auroc(ka, A["unk"][sn][g]) - auroc(kb_, B["unk"][sn][g]) for g in fidx])
        ua, ub = A["unk"][sn][fidx], B["unk"][sn][fidx]
        return metric(ka, ua, uf[fidx]) - metric(kb_, ub, uf[fidx])
    pt = np.nanmean([diff(s, t) for s in splits for t in seeds]); rng = np.random.RandomState(seed); bs = []
    for _ in range(n_boot):
        fb = rng.choice(fams, len(fams)); vals = []
        for s in splits:
            uf = SC[(ds, s, seeds[0], "agef_sharp")]["unk_fam"]
            groups = [np.where(uf == f)[0] for f in fb if (uf == f).any()]
            if not groups: continue
            idx = groups if macro else np.concatenate(groups)
            n_te = len(SC[(ds, s, seeds[0], "agef_sharp")]["sc"]["test"]["energy"]); kb = rng.randint(0, n_te, n_te)
            vals += [diff(s, t, idx, kb) for t in rng.choice(seeds, len(seeds))]
        vals = [v for v in vals if not np.isnan(v)]
        if vals: bs.append(np.mean(vals))
    return pt, np.array(bs)
COMP = [("G1", "seedens_agef_sharp", "deep_ensemble", "energy"), ("G1", "seedens_agef_sharp", "deep_ensemble", "knn50"),
        ("G1", "seedens_agef_sharp", "seedens_mobilenet", "energy"), ("G1", "seedens_agef_tuned", "deep_ensemble", "energy"),
        ("G1", "seedens_agef_sharp", "agef_sharp", "energy"), ("G2", "seedens_mobilenet", "deep_ensemble", "energy"),
        ("G2", "seedens_mobilenet", "deep_ensemble", "knn50"),
        ("G3", "stack_lr", "deep_ensemble_halfcal", "energy"), ("G3", "stack_weights", "deep_ensemble_halfcal", "energy"),
        ("G3", "stack_lr", "agef_sharp", "energy"), ("G3", "frozen_gate", "deep_ensemble_halfcal", "energy"), ("G3", "frozen_gate", "agef_sharp", "energy"),
        ("C1", "deep_ensemble", "agef_sharp", "energy"), ("C2", "deep_ensemble", "agef_sharp", "knn50"), ("C3", "avg_joint", "agef_sharp", "energy"),
        ("C4", "agef_tuned", "agef_sharp", "energy"), ("C5", "agef_tuned", "deep_ensemble", "energy"), ("C6", "deep_ensemble", "best_single", "energy")]
crow = []
for gid, a, b, sn in COMP:
    per = {ds: boot(ds, a, b, sn, N_BOOT, zlib.crc32(f"{a}{b}{sn}".encode()) % 1000) for ds in RUNS}
    for ds, (pt, bs) in per.items():
        crow.append(dict(group=gid, config=a, reference=b, score=sn, dataset=ds, delta=pt, se=bs.std(ddof=1),
                         ci95_lo=np.percentile(bs, 2.5), ci95_hi=np.percentile(bs, 97.5), ci90_lo=np.percentile(bs, 5), ci90_hi=np.percentile(bs, 95)))
    nmin = min(len(v[1]) for v in per.values()); pb = np.mean([v[1][:nmin] for v in per.values()], 0); pt = np.mean([v[0] for v in per.values()])
    # G8: random effects (DerSimonian-Laird) from per-dataset estimates and bootstrap SEs
    est = np.array([v[0] for v in per.values()]); se = np.array([v[1].std(ddof=1) for v in per.values()]); w = 1 / se ** 2
    fe = (w * est).sum() / w.sum(); Q = (w * (est - fe) ** 2).sum(); tau2 = max(0.0, (Q - (len(est) - 1)) / (w.sum() - (w ** 2).sum() / w.sum()))
    wr = 1 / (se ** 2 + tau2); re = (wr * est).sum() / wr.sum(); re_se = np.sqrt(1 / wr.sum())
    crow.append(dict(group=gid, config=a, reference=b, score=sn, dataset="pooled", delta=pt, se=pb.std(ddof=1),
                     ci95_lo=np.percentile(pb, 2.5), ci95_hi=np.percentile(pb, 97.5), ci90_lo=np.percentile(pb, 5), ci90_hi=np.percentile(pb, 95),
                     random_effects=re, re_ci_lo=re - 1.96 * re_se, re_ci_hi=re + 1.96 * re_se, tau2=tau2,
                     mde80=(norm.ppf(0.975) + norm.ppf(0.8)) * pb.std(ddof=1)))
    say(f"{gid} {a} vs {b} ({sn}) done")
CMP = pd.DataFrame(crow)
CMP["tost_equivalent"] = (CMP.ci90_lo > -MARGIN) & (CMP.ci90_hi < MARGIN)
CMP["ci95_excludes_0"] = (CMP.ci95_lo > 0) | (CMP.ci95_hi < 0)
CMP.to_csv(f"{OUT}/G_comparisons.csv", index=False)

# ----------------------------------------------------------------------------- G4 gate collapse
GT = pd.DataFrame(AUX["gate"]); GT["collapsed_gate"] = GT.max_mean_weight > 0.9
GT.to_csv(f"{OUT}/G4_gate_per_run.csv", index=False)

# ----------------------------------------------------------------------------- G6 nested shortcut (select on split s', evaluate on split s)
POOL = [(m, sn) for m in ["agef_sharp", "agef_tuned", "avg_joint", "deep_ensemble"] + SINGLES for sn in ["energy", "knn50"]]
nrow = []
for ds in RUNS:
    g = L[(L.dataset == ds) & (L.seed >= 0)]; g = g[g.apply(lambda r: (r.model, r.score) in POOL, axis=1)]
    piv = g.groupby(["model", "score", "split"]).ufr5.mean().unstack("split"); ref = piv.loc[("deep_ensemble", "energy")]
    nested = [piv.loc[piv[s2].idxmax(), s] for s in piv.columns for s2 in piv.columns if s2 != s]
    nrow.append(dict(dataset=ds, pool=len(piv), prespecified=ref.mean(), pool_mean=piv.values.mean(), nested_select_other_split=np.mean(nested),
                     same_split_best=piv.max(0).mean(), nested_vs_prespecified_pp=100 * (np.mean(nested) - ref.mean()),
                     same_split_vs_prespecified_pp=100 * (piv.max(0).mean() - ref.mean())))
NS = pd.DataFrame(nrow); NS.to_csv(f"{OUT}/G6_nested_shortcuts.csv", index=False)

# ----------------------------------------------------------------------------- G9 near-twin calibration images
POP = np.array([bin(i).count("1") for i in range(256)], np.uint8); trow = []
class FB:
    def __init__(self, classes, targets): self.classes = classes; self.samples = [(f"x{i}", int(t)) for i, t in enumerate(targets)]; self.targets = list(targets)
for ds, R in RUNS.items():
    keys = json.load(open(f"{R['dir']}/image_keys.json"))["keys"]; hb = np.stack([np.frombuffer(bytes.fromhex(k[2]), dtype=np.uint8) for k in keys])
    kt = tuple(ax.CFG["GROUP_KEYS"]) + ((2,) if ds in ax.CFG.get("IDENTICAL_WHOLE_DATASETS", ()) else ())
    with contextlib.redirect_stdout(io.StringIO()):
        G = ax.compute_groups(FB(R["classes"], R["targets"]), f"{R['dir']}/image_keys.json", kt, 0.9, 10, 8, 250)
    sizes = np.bincount(G["groups"])
    for s in R["splits"]:
        sp = json.load(open(f"{R['dir']}/split{s}/split.json")); tr = np.array(sp["train"])
        for p in ("calib", "test"):
            q = np.array(sp[p]); d = POP[hb[q][:, None] ^ hb[tr][None]].sum(-1); dm = d.min(1); twin = tr[d.argmin(1)]; m = dm <= 8
            trow.append(dict(dataset=ds, split=s, part=p, n=len(q), near_twins=int(m.sum()),
                             twin_group_at_cap=int((sizes[G["groups"][twin[m]]] >= 250).sum()) if m.any() else 0,
                             own_group_at_cap=int((sizes[G["groups"][q[m]]] >= 250).sum()) if m.any() else 0,
                             families=",".join(sorted(set(R["classes"][c] for c in R["targets"][q[m]]))) if m.any() else ""))
TW = pd.DataFrame(trow); TW.to_csv(f"{OUT}/G9_near_twins.csv", index=False)

# ----------------------------------------------------------------------------- G10 original MaleVis run without VBA in evaluation
d4 = find("auditx_final_malevis"); g10 = []
if d4:
    man = json.load(open(f"{d4}/MANIFEST.json")); assert man["protocol"] == V4
    tg = np.load(f"{d4}/targets.npy"); cl = json.load(open(f"{d4}/classes.json")); vba = cl.index("VBA") if "VBA" in cl else -1
    for s in sorted(int(p.split("split")[-1]) for p in glob.glob(f"{d4}/split*") if os.path.isdir(p)):
        sp = json.load(open(f"{d4}/split{s}/split.json"))
        keep = {p: tg[np.array(sp[p])] != vba for p in ("calib", "test", "unk")}
        for t in man["args"]["train_seeds"]:
            outs = {m: load_npz(f"{d4}/split{s}/seed{t}/{m}.npz")[0] for m in ["agef_sharp", "avg_joint"] + SINGLES}
            cf = {"deep_ensemble": ens([outs[x] for x in SINGLES]), "agef_sharp": outs["agef_sharp"], "avg_joint": outs["avg_joint"]}
            for m, o in cf.items():
                e = {p: energy(o[p]["logits"]) for p in ("calib", "test", "unk")}
                for ver, kp in (("all", None), ("without_VBA", keep)):
                    ee = e if kp is None else {p: e[p][kp[p]] for p in e}
                    th = cthr(ee["calib"], 0.05)
                    g10.append(dict(split=s, seed=t, model=m, version=ver, auroc=auroc(ee["test"], ee["unk"]), ufr5=(ee["unk"] > th).mean(), far5=(ee["test"] > th).mean()))
    G10 = pd.DataFrame(g10).groupby(["model", "version"]).mean(numeric_only=True).drop(columns=["split", "seed"]).reset_index()
    G10.to_csv(f"{OUT}/G10_original_malevis_without_vba.csv", index=False)
else:
    say("G10 skipped: original MaleVis run (auditx_final_malevis) not attached"); G10 = pd.DataFrame()

# ----------------------------------------------------------------------------- G11 C1-C6 under family-macro AUROC
mrow = []
for gid, a, b, sn in [c for c in COMP if c[0].startswith("C")]:
    per = {ds: boot(ds, a, b, sn, N_BOOT, zlib.crc32(f"macro{a}{b}{sn}".encode()) % 1000, macro=True) for ds in RUNS}
    nmin = min(len(v[1]) for v in per.values()); pb = np.mean([v[1][:nmin] for v in per.values()], 0)
    for ds, (pt, bs) in list(per.items()) + [("pooled", (np.mean([v[0] for v in per.values()]), pb))]:
        mrow.append(dict(id=gid, config=a, reference=b, score=sn, dataset=ds, delta_family_macro=pt, ci95_lo=np.percentile(bs, 2.5), ci95_hi=np.percentile(bs, 97.5)))
    say(f"G11 {gid} family-macro done")
FM = pd.DataFrame(mrow); FM.to_csv(f"{OUT}/G11_family_macro_comparisons.csv", index=False)

# ----------------------------------------------------------------------------- G12 distinct families and leave-one-family-out jackknife (point estimates)
jrow = []
for ds in RUNS:
    seeds = RUNS[ds]["seeds"]; fams = sorted({f for s in RUNS[ds]["splits"] for f in SC[(ds, s, seeds[0], "agef_sharp")]["unk_fam"]})
    jrow.append(dict(dataset=ds, id="n_families", value=len(fams)))
    for gid, a, b, sn in [c for c in COMP if c[0].startswith("C")]:
        full = boot(ds, a, b, sn, 0, 0)[0]
        jk = [boot(ds, a, b, sn, 0, 0, drop_family=f)[0] for f in fams]
        jrow.append(dict(dataset=ds, id=gid, value=full, jk_min=np.nanmin(jk), jk_max=np.nanmax(jk), sign_changes=int(np.sum(np.sign(jk) != np.sign(full)))))
JK = pd.DataFrame(jrow); JK.to_csv(f"{OUT}/G12_family_jackknife.csv", index=False)

# ----------------------------------------------------------------------------- G13 what +-0.03 AUROC means operationally
pr = L[(L.seed >= 0) & L.score.isin(["energy", "knn50"]) & ~L.model.isin(["stack_lr", "stack_weights", "frozen_gate", "deep_ensemble_halfcal"])]
slope = np.polyfit(pr.auroc, pr.ufr_matched5, 1)[0]
sd_seed = L[(L.seed >= 0)].groupby(["dataset", "model", "score", "split"]).auroc.std().median()
json.dump(dict(slope_ufr_matched5_per_auroc=slope, ufr_pp_for_0p03_auroc=100 * 0.03 * slope, median_seed_sd_auroc=sd_seed),
          open(f"{OUT}/G13_margin_meaning.json", "w"), indent=1)
say(f"G13: 0.03 AUROC corresponds to about {100 * 0.03 * slope:.1f} pp UFR at matched 5% FAR; median seed SD of AUROC {sd_seed:.3f}")

# ----------------------------------------------------------------------------- G14 KNN bank-size sensitivity (deep ensemble, AGEF-Net; first seed)
krow = []
for ds, R in RUNS.items():
    for s in R["splits"]:
        t = R["seeds"][0]; outs = {m: load_npz(f"{R['dir']}/split{s}/seed{t}/{m}.npz")[0] for m in ["agef_sharp"] + SINGLES}
        cf = {"deep_ensemble": ens([outs[x] for x in SINGLES]), "agef_sharp": outs["agef_sharp"]}
        for m, o in cf.items():
            for cap in (1000, 3000, 6000):
                kn = ax.KNNScorer(o["train"]["feats"], 50, cap, s)
                krow.append(dict(dataset=ds, split=s, model=m, bank=cap, auroc=auroc(kn(o["test"]["feats"]), kn(o["unk"]["feats"]))))
KC = pd.DataFrame(krow).groupby(["dataset", "model", "bank"]).auroc.mean().unstack("bank"); KC.to_csv(f"{OUT}/G14_knn_bank_size.csv")

# ----------------------------------------------------------------------------- G15 tuned-AGEF learning-rate sweep
lrrow = []
for ds, R in RUNS.items():
    f = f"{R['dir']}/agef_tuned_lr.json"
    if os.path.exists(f):
        j = json.load(open(f))
        for r in j["sweep"]: lrrow.append(dict(dataset=ds, lr=r["lr"], val_acc=r["val_acc"], val_nll=r["val_nll"], chosen=r["lr"] == j["chosen"]))
LRS = pd.DataFrame(lrrow); LRS.to_csv(f"{OUT}/G15_lr_sweep.csv", index=False)
FG = pd.DataFrame(AUX.get("fgate", [])); FG.to_csv(f"{OUT}/G3b_frozen_gate_weights.csv", index=False)

# ----------------------------------------------------------------------------- G16 near-duplicate rule validity (needs the image datasets)
DATA = {"Malimg": "Malimg_Dataset_25classes", "MaleVis": "MaleVis", "Dumpware10": "dumpware10_images"}
def merge_dumpware(raw, merged="/kaggle/working/dumpware10_merged"):
    root = next((c for c in [os.path.join(raw, "224", "224"), os.path.join(raw, "224"), raw]
                 if os.path.isdir(c) and {"TRAINING", "TEST"} <= set(os.listdir(c))), None)
    if root is None: return raw
    for split in ["TRAINING", "TEST"]:
        for cls in sorted(os.listdir(os.path.join(root, split))):
            sd = os.path.join(root, split, cls)
            if not os.path.isdir(sd): continue
            dd = os.path.join(merged, cls); os.makedirs(dd, exist_ok=True)
            for fn in os.listdir(sd):
                dst = os.path.join(dd, f"{split.lower()}__{fn}")
                if not os.path.lexists(dst): os.symlink(os.path.join(sd, fn), dst)
    return merged
vrow, grid = [], []
try:
    from PIL import Image
    from torchvision import datasets as tvd
    for ds, R in RUNS.items():
        hits = [h for h in glob.glob(f"{ROOT}/**/{DATA[ds]}", recursive=True) if os.path.isdir(h)]
        if not hits: say(f"G16 {ds}: dataset not attached - skipped"); continue
        path = merge_dumpware(hits[0]) if ds == "Dumpware10" else hits[0]
        base = tvd.ImageFolder(path); assert base.classes == R["classes"], f"{ds}: class order differs"
        keys = json.load(open(f"{R['dir']}/image_keys.json"))["keys"]; hb = np.stack([np.frombuffer(bytes.fromhex(k[2]), dtype=np.uint8) for k in keys])
        tg = R["targets"]; rng = np.random.RandomState(0)
        def img(i):
            with Image.open(base.samples[i][0]) as im: return np.asarray(im.convert("L").resize((64, 64), Image.BILINEAR), dtype=np.float64)
        def ncc(i, j):
            a, b = img(i), img(j); a = (a - a.mean()) / (a.std() + 1e-9); b = (b - b.mean()) / (b.std() + 1e-9); return float((a * b).mean())
        pairs = {"0 bits": [], "1-8 bits": [], "9-16 bits": [], "random same family": []}
        for c in np.unique(tg):
            idx = np.where(tg == c)[0]
            if len(idx) < 2: continue
            sub = rng.choice(idx, min(len(idx), 300), replace=False)
            d = POP[hb[sub][:, None] ^ hb[sub][None]].sum(-1)
            for lo, hi, nm in [(0, 0, "0 bits"), (1, 8, "1-8 bits"), (9, 16, "9-16 bits")]:
                ii, jj = np.where((d >= lo) & (d <= hi) & (np.triu(np.ones_like(d), 1) > 0))
                pairs[nm] += [(sub[a_], sub[b_]) for a_, b_ in zip(ii, jj)]
            pairs["random same family"] += [tuple(rng.choice(idx, 2, replace=False)) for _ in range(5)]
        for nm, pl in pairs.items():
            if not pl: continue
            pick = [pl[k] for k in rng.choice(len(pl), min(len(pl), 150), replace=False)]
            v = np.array([ncc(i, j) for i, j in pick])
            vrow.append(dict(dataset=ds, pair_type=nm, n_pairs_available=len(pl), n_checked=len(pick), ncc_median=np.median(v),
                             share_ncc_ge_0p9=float((v >= 0.9).mean()), share_ncc_ge_0p8=float((v >= 0.8).mean())))
            if nm == "1-8 bits": grid += [(ds, base.samples[i][0], base.samples[j][0]) for i, j in pick[:8]]
        say(f"G16 {ds} done")
    if grid:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        fig, axs = plt.subplots(len(grid), 2, figsize=(4, 2 * len(grid)))
        for r_, (ds, p1, p2) in enumerate(grid):
            for c_, pp in enumerate((p1, p2)):
                with Image.open(pp) as im: axs[r_, c_].imshow(im.convert("L"), cmap="gray")
                axs[r_, c_].axis("off"); axs[r_, c_].set_title(ds if c_ == 0 else "", fontsize=6)
        plt.tight_layout(); plt.savefig(f"{OUT}/G16_pairs_1to8bits.png", dpi=110); plt.close()
except Exception as e:
    say("G16 failed:", repr(e))
VAL = pd.DataFrame(vrow); VAL.to_csv(f"{OUT}/G16_near_duplicate_validity.csv", index=False)

# ----------------------------------------------------------------------------- summary
say("=" * 100)
P = CMP[CMP.dataset == "pooled"]
for r in P.itertuples():
    say(f"{r.group} {r.config} vs {r.reference} ({r.score}): {r.delta:+.3f} [{r.ci95_lo:+.3f}, {r.ci95_hi:+.3f}] 90% [{r.ci90_lo:+.3f}, {r.ci90_hi:+.3f}]"
        f" TOST {'equivalent' if r.tost_equivalent else 'not shown'} | random effects {r.random_effects:+.3f} [{r.re_ci_lo:+.3f}, {r.re_ci_hi:+.3f}] | MDE80 {r.mde80:.3f}")
say("Gate collapse (runs with one expert weight > 0.9): " + GT.groupby(["dataset", "model"]).collapsed_gate.sum().to_string())
say("Nested shortcuts:\n" + NS.round(3).to_string(index=False))
say("Family-macro C1-C6 (pooled):\n" + FM[FM.dataset == "pooled"].round(3).to_string(index=False))
say("Family jackknife:\n" + JK.round(3).to_string(index=False))
say("KNN bank size:\n" + KC.round(3).to_string())
if len(LRS): say("LR sweep:\n" + LRS.round(3).to_string(index=False))
if len(VAL): say("Near-duplicate validity:\n" + VAL.round(3).to_string(index=False))
if len(G10): say("Original MaleVis without VBA (energy):\n" + G10.round(3).to_string(index=False))
say("=" * 100)
open(f"{OUT}/G_SUMMARY.txt", "w").write("\n".join(LOG))
print("outputs:", sorted(os.listdir(OUT)))

In [ ]:
import subprocess, sys
p = subprocess.Popen([sys.executable, "/kaggle/working/auditx_review4.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd="/kaggle/working")
for line in p.stdout: print(line, end="", flush=True)
p.wait(); assert p.returncode == 0, "analysis failed - send the log above"

In [ ]:
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 300)
O = "/kaggle/working/auditx_review4"
print(open(f"{O}/G_SUMMARY.txt").read()[-6000:])
for f in ["G6_nested_shortcuts.csv", "G9_near_twins.csv", "G10_original_malevis_without_vba.csv", "G11_family_macro_comparisons.csv", "G12_family_jackknife.csv", "G16_near_duplicate_validity.csv"]:
    try: print("\n###", f); display(pd.read_csv(f"{O}/{f}").round(4))
    except FileNotFoundError: print("not produced")